In [1]:
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier



In [2]:
train_df = pd.read_csv('../data/train_cleaned.csv')
test_df = pd.read_csv('../data/test_cleaned.csv')

In [3]:
test_df = test_df.drop('id', axis=1)

In [4]:
Xd = train_df.drop('price_range', axis=1)
yd = train_df['price_range']

DT model

In [5]:
test_sizes = [0.1, 0.15, 0.2, 0.25, 0.3]
max_depths = range(1, 21)

results = []

for test_size in test_sizes:

    sss = StratifiedShuffleSplit(
        n_splits=10,
        test_size=test_size,
        random_state=42
    )

    for depth in max_depths:

        train_scores = []
        test_scores = []

        for train_idx, test_idx in sss.split(Xd, yd):

            X_train = Xd.iloc[train_idx]
            X_test = Xd.iloc[test_idx]

            y_train = yd.iloc[train_idx]
            y_test = yd.iloc[test_idx]

            clf = DecisionTreeClassifier(
                max_depth=depth,
                random_state=42
            )

            clf.fit(X_train, y_train)

            train_pred = clf.predict(X_train)
            test_pred = clf.predict(X_test)

            train_scores.append(
                accuracy_score(y_train, train_pred)
            )

            test_scores.append(
                accuracy_score(y_test, test_pred)
            )

        results.append({
            'Test_Size': test_size,
            'Max_Depth': depth,
            'Train_Acc_Mean': np.mean(train_scores),
            'Test_Acc_Mean': np.mean(test_scores),
            'Train_Acc_STD': np.std(train_scores),
            'Test_Acc_STD': np.std(test_scores),
            'Gap': np.mean(train_scores) - np.mean(test_scores)
        })

df_results = pd.DataFrame(results)

df_results = df_results.sort_values(
    by='Test_Acc_Mean',
    ascending=False
)

df_results.head(20)

,Test_Size,Max_Depth,Train_Acc_Mean,Test_Acc_Mean,Train_Acc_STD,Test_Acc_STD,Gap
47,0.20,8,0.977388,0.848077,0.004512,0.013629,0.129311
48,0.20,9,0.989003,0.846429,0.002900,0.014817,0.142575
46,0.20,7,0.957045,0.845604,0.006442,0.013221,0.111440
49,0.20,10,0.995189,0.844780,0.002516,0.015936,0.150409
5,0.10,6,0.924068,0.844505,0.003756,0.017033,0.079563
52,0.20,13,0.999931,0.843681,0.000206,0.014033,0.156250
55,0.20,16,1.000000,0.843132,0.000000,0.014352,0.156868
53,0.20,14,1.000000,0.843132,0.000000,0.014352,0.156868
54,0.20,15,1.000000,0.843132,0.000000,0.014352,0.156868
57,0.20,18,1.000000,0.843132,0.000000,0.014352,0.156868


best model is: max depth = 8 and test size = 0.2

In [6]:
X_train, X_test, y_train, y_test = train_test_split(Xd, yd, test_size=0.2, random_state=42)

best_DT_clf = DecisionTreeClassifier(max_depth=8, random_state=42)
best_DT_clf.fit(X_train, y_train)
print(best_DT_clf.score(X_train, y_train))
print(best_DT_clf.score(X_test, y_test))


0.9786941580756013
0.8351648351648352


check the trained model is predict well.

In [7]:
y_new_pred = best_DT_clf.predict(test_df)
pd.Series(y_new_pred).value_counts(normalize=True)

3    0.264108
2    0.256208
0    0.244921
1    0.234763
Name: proportion, dtype: float64

In [8]:
probs = best_DT_clf.predict_proba(test_df)

max_probs = probs.max(axis=1)
print(max_probs.mean())

0.9775114802662427


RF model

In [10]:


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8, 10],
    'min_samples_leaf': [2, 4, 8],
    'min_samples_split': [5, 10, 20]
}

results = []

for n_trees in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for leaf in param_grid['min_samples_leaf']:
            for split in param_grid['min_samples_split']:
                train_scores = []
                test_scores = []

                for train_idx, test_idx in sss.split(Xd, yd):
                    X_train, X_test = Xd.iloc[train_idx], Xd.iloc[test_idx]
                    y_train, y_test = yd.iloc[train_idx], yd.iloc[test_idx]

                    rf_clf = RandomForestClassifier(
                        n_estimators=n_trees,
                        max_depth=depth,
                        min_samples_leaf=leaf,
                        min_samples_split=split,
                        random_state=42,
                        n_jobs=-1
                    )
                    rf_clf.fit(X_train, y_train)

                    train_scores.append(rf_clf.score(X_train, y_train))
                    test_scores.append(rf_clf.score(X_test, y_test))

                results.append({
                    'n_estimators': n_trees,
                    'max_depth': depth,
                    'min_samples_leaf': leaf,
                    'min_samples_split': split,
                    'Train_Acc_Mean': np.mean(train_scores),
                    'Test_Acc_Mean': np.mean(test_scores),
                    'Test_Acc_STD': np.std(test_scores),
                    'Gap': np.mean(train_scores) - np.mean(test_scores)
                })

df_rf_tuned = pd.DataFrame(results)
df_rf_tuned.sort_values(by=['Test_Acc_Mean', 'Gap'], ascending=[False, True]).head(10)

,n_estimators,max_depth,min_samples_leaf,min_samples_split,Train_Acc_Mean,Test_Acc_Mean,Test_Acc_STD,Gap
66,200,10,4,5,0.990447,0.880495,0.016902,0.109952
30,100,10,4,5,0.990378,0.879945,0.012592,0.110433
63,200,10,2,5,0.997526,0.878297,0.018308,0.119229
64,200,10,2,10,0.990790,0.876923,0.021450,0.113867
67,200,10,4,10,0.987766,0.873626,0.015589,0.114140
27,100,10,2,5,0.997113,0.870330,0.016428,0.126784
18,100,8,2,5,0.986735,0.868681,0.020588,0.118054
28,100,10,2,10,0.990790,0.868407,0.015707,0.122384
68,200,10,4,20,0.974639,0.867857,0.017653,0.106782
65,200,10,2,20,0.977251,0.867857,0.018972,0.109394


best model is n_estimators = 100, max_depth = 10, min_samples_leaf = 4, min_samples_split = 5

In [11]:
best_RF_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=4,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

best_RF_clf.fit(Xd, yd)

test_predictions = best_RF_clf.predict(test_df)

In [13]:
rf_preds = best_RF_clf.predict(test_df)
pd.Series(rf_preds).value_counts(normalize=True).sort_index() * 100

0    25.507901
1    24.492099
2    23.589165
3    26.410835
Name: proportion, dtype: float64

In [19]:
rf_probs = best_RF_clf.predict_proba(test_df)
max_probs = rf_probs.max(axis=1)
print(f"{max_probs.mean() * 100:.2f}%")
print(f"predict more than 70%: {(max_probs >= 0.70).mean() * 100:.2f}%")

58.15%
predict more than 70%: 20.54%


In [16]:
dt_preds = best_DT_clf.predict(test_df)
agreement = (rf_preds == dt_preds).mean() * 100
print(f"DT vs RF: {agreement:.2f}%")

DT vs RF: 85.89%
